In [1]:
from datetime import datetime
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from datetime import datetime
import os
from pathlib import Path
import fsspec

import re
pd.set_option('display.max_columns', None)
sys.path.insert(0, os.path.abspath('../..'))
import repo_paths  # noqa: F401
%load_ext autoreload
%autoreload 2
from loading_runs import load_results_from_pickle, compute_rowwise_metrics, plot_mask_distributions, plot_metric_vs_lambda

### q sensitivity

Single seed of Hide&Seek across Syn1-6

In [78]:
temp = load_results_from_pickle(folder='ICML_experiments/annealing',
                                        time_threshold='25_09_03_12_00_00',
                                        keep_run_type=None,
                                          is_output=False,
                                  numeric_cols = ['lmbda','accuracy', 'roc_auc', 'pct_sig','TPR_mean','FDR_mean','f1','pr_auc']
                                        )

assert (temp.num_syn_features == 11).all()

20 seeds for each Syn for each lmbda_exponent

In [79]:
temp.groupby(['lmbda','syn','lmbda_exponent'])[['TPR_mean','FDR_mean','f1']].count()

TPR_mean  FDR_mean  f1
lmbda syn  lmbda_exponent                        
0.3   Syn1 0.5                   20        20  20
           1.0                   20        20  20
           2.0                   20        20  20
           3.0                   20        20  20
           4.0                   20        20  20
           5.0                   20        20  20
      Syn2 0.5                   20        20  20
           1.0                   20        20  20
           2.0                   20        20  20
           3.0                   20        20  20
           4.0                   20        20  20
           5.0                   20        20  20
      Syn3 0.5                   20        20  20
           1.0                   20        20  20
           2.0                   20        20  20
           3.0                   20        20  20
           4.0                   20        20  20
           5.0                   20        20  20
      Syn4 0.5                   20        20  20
           1.0                   20        20  20
           2.0                   20        20  20
           3.0                   20        20  20
           4.0                   20        20  20
           5.0                   20        20  20
      Syn5 0.5                   20        20  20
           1.0                   20        20  20
           2.0                   20        20  20
           3.0                   20        20  20
           4.0                   20        20  20
           5.0                   20        20  20
      Syn6 0.5                   20        20  20
           1.0                   20        20  20
           2.0                   20        20  20
           3.0                   20        20  20
           4.0                   20        20  20
           5.0                   20        20  20

In [80]:
a = temp.groupby(['model','lmbda','lmbda_exponent'])[['TPR_mean','FDR_mean','f1','roc_auc','pr_auc','pct_sig']]
a = a.mean().sort_values(by='lmbda_exponent',ascending=True).reset_index()
a = a[['lmbda_exponent','TPR_mean','FDR_mean','f1','roc_auc']].round(2)
a

,lmbda_exponent,TPR_mean,FDR_mean,f1,roc_auc
0,0.5,38.34,5.33,44.45,0.68
1,1.0,78.43,2.71,82.88,0.79
2,2.0,97.66,2.49,97.26,0.83
3,3.0,98.53,4.30,96.67,0.83
4,4.0,98.78,5.44,96.13,0.83
5,5.0,98.88,6.65,95.44,0.83


In [49]:
# 1. Rename the columns using a dictionary mapping
a = a.rename(columns={
    'lmbda_exponent': 'q',
    'TPR_mean': 'TPR',
    'FDR_mean': 'FDR',
    'f1': 'F1',
    'roc_auc': 'AUROC'
})

# 2. Convert to LaTeX and print (dropping the numeric index for a cleaner table)
print(a.to_latex(index=False))

\begin{tabular}{rrrrr}
\toprule
  q &   TPR &  FDR &    F1 &  AUROC \\
\midrule
0.5 & 38.34 & 5.33 & 44.45 &   0.68 \\
1.0 & 78.43 & 2.71 & 82.88 &   0.79 \\
2.0 & 97.66 & 2.49 & 97.26 &   0.83 \\
3.0 & 98.53 & 4.30 & 96.67 &   0.83 \\
4.0 & 98.78 & 5.44 & 96.13 &   0.83 \\
5.0 & 98.88 & 6.65 & 95.44 &   0.83 \\
\bottomrule
\end{tabular}



/tmp/ipykernel_433932/3188238717.py:11: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(a.to_latex(index=False))
